
## Creating a Function Tool

In [6]:
from llama_index.core.tools import FunctionTool

def get_car_price(car: str) -> str:
    """Useful for getting a price of a car"""
    print(f"Getting price of{car}")
    return f"The price of {car} is $10,000"

tool = FunctionTool.from_defaults(
    get_car_price,
    name="my_car_price_tool",
    description="Useful for getting a price of a car",
)

response = tool.call("Toyota Landcruiser")

response.blocks[0].text

Getting price ofToyota Landcruiser


'The price of Toyota Landcruiser is $10,000'

## Creating a QueryEngineTool

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

embed_model = HuggingFaceEmbedding("BAAI/bge-small-en-v1.5")

db = chromadb.PersistentClient(path="./abv_flights_db")

chroma_collection = db.get_or_create_collection("flights")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_vector_store(vector_store, 
                                           embed_model=embed_model
                                          )

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

query_engine = index.as_query_engine(llm=llm)

tool = QueryEngineTool.from_defaults(query_engine, 
                                     name="my_car_price_tool", 
                                     description="Useful for getting a price of a car"
                                    )

## Creating Toolspecs

In [5]:
!uv add llama-index-tools-google

Resolved 390 packages in 16ms
Audited 363 packages in 93ms


In [8]:
from llama_index.tools.google import GmailToolSpec

tool_spec = GmailToolSpec()

tool_spec_list = tool_spec.to_tool_list()

[(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list][2]

('create_draft',
 "create_draft(to: Optional[List[str]] = None, subject: Optional[str] = None, message: Optional[str] = None) -> str\n\n        Create and insert a draft email.\n           Print the returned draft's message and id.\n           Returns: Draft object, including draft id and message meta data.\n\n        Args:\n            to (Optional[str]): The email addresses to send the message to\n            subject (Optional[str]): The subject for the event\n            message (Optional[str]): The message for the event")

## Model Context Protocol (MCP) in LlamaIndex


In [10]:
!uv add llama-index-tools-mcp

Resolved 394 packages in 27.18s                                      
Prepared 8 packages in 20m 56s                                           
Uninstalled 7 packages in 238ms
Installed 11 packages in 108ms                              
 - arize-phoenix==20.3.0
 + arize-phoenix==19.8.0
 - blaxel==0.4.13
 + blaxel==0.3.5
 - fastmcp-slim==3.4.8
 + fastmcp-slim==4.1.2
 + llama-index-tools-mcp==0.6.0
 - mcp==1.30.0
 + mcp==2.3.0
 + mcp-types==2.3.0
 - pydantic-monty==0.0.19
 + pydantic-monty==1.2.0
 + pydantic-monty-client==1.2.0
 - pydantic-monty-runtime==0.0.19
 + pydantic-monty-runtime==1.2.0
 + sqlean-py==3.50.4.5
 - websockets==16.1.1
 + websockets==15.0.1


In [ ]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")

mcp_tool = McpToolSpec(client=mcp_client)

agent = await get_agent(mcp_tool)

agent_context = Context(agent)

## Utility Tools

In [12]:
!uv add llama-index-readers-wikipedia

Resolved 395 packages in 32ms
Audited 366 packages in 57ms


In [ ]:
from llama_index.readers.wikipedia import WikipediaReader

from llama_index.core.tools.ondemand_loader_tool import OnDemandLoaderTool

reader = WikipediaReader()

tool = OnDemandLoaderTool.from_defaults(
    reader,
    name="Wikipedia Tool",
    description="A tool for loading data and querying articles from Wikipedia",
)

In [ ]:

from llama_index.core.tools.tool_spec.load_and_search import (
    LoadAndSearchToolSpec,
)

from llama_index.core.agent.workflow import FunctionAgent

from llama_index.tools.wikipedia import WikipediaToolSpec


wiki_spec = WikipediaToolSpec()

tool = wiki_spec.to_tool_list()[1]

agent = FunctionAgent(
    llm=llm, tools=LoadAndSearchToolSpec.from_defaults(tool).to_tool_list()
)